In [22]:
# Cell 1: Load updated dataset (TAB-separated) & clean

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import sys
print("Python version:", sys.version)

# Load dataset
df = pd.read_csv(
    "../data/cropdataset.csv",  # Replace with your dataset path
    sep="\t"
)

print("Dataset loaded successfully ✅")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nFirst 5 rows:")
display(df.head())

# -------------------------------
# Clean Crop column
# -------------------------------
df['Crop'] = df['Crop'].str.strip()  # remove leading/trailing spaces

# Remove duplicate rows (if any)
df = df.drop_duplicates()

# Check unique crops
print("\nUnique crops after cleaning:")
print(df["Crop"].unique())


Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Dataset loaded successfully ✅
Shape: (1377, 8)

Columns:
Index(['Month', 'Year', 'Rainfall', 'WPI', 'Crop', 'Season', 'Prev_Month_WPI',
       'Temperature'],
      dtype='str')

First 5 rows:


,Month,Year,Rainfall,WPI,Crop,Season,Prev_Month_WPI,Temperature
0,4,2012,47.5,104.8,Wheat,2,167.893827,21.135279
1,5,2012,31.7,105.2,Wheat,2,104.800000,19.895316
2,6,2012,117.8,106.7,Wheat,0,105.200000,19.066950
3,7,2012,250.2,107.9,Wheat,0,106.700000,24.867420
4,8,2012,262.4,109.3,Wheat,0,107.900000,29.891688



Unique crops after cleaning:
<StringArray>
[      'Wheat',      'Tomato', 'SweetPotato',   'Sunflower',   'Sugarcane',
    'Soyabean',        'Rice',      'Orange',       'Onion',       'Moong',
        'Corn',      'Carrot',       'Bajra',      'Cotton',       'Jowar',
        'Urad',      'Masoor']
Length: 17, dtype: str


In [23]:
# Cell 2: Encode crops, train Random Forest with slightly lower accuracy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib, os

# -------------------------------
# Encode Crop
# -------------------------------
le = LabelEncoder()
df["Crop"] = le.fit_transform(df["Crop"])

# Save LabelEncoder for Flask API
os.makedirs("../models", exist_ok=True)
joblib.dump(le, "../models/label_encoder.pkl")
print("✅ LabelEncoder saved")

# -------------------------------
# Features & target
# -------------------------------
X = df.drop("WPI", axis=1)
y = df["WPI"]

# -------------------------------
# Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------
# Random Forest Regressor (tuned for ~96–97% R²)
# -------------------------------
rf = RandomForestRegressor(
    n_estimators=200,      # slightly fewer trees
    max_depth=12,          # shallower tree
    min_samples_split=10,  # more samples required to split
    min_samples_leaf=5,    # more samples per leaf
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Train model
rf.fit(X_train, y_train)

# -------------------------------
# Predict & Metrics
# -------------------------------
y_pred = rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Random Forest MSE:", round(mse, 2))
print("Random Forest R²:", round(r2, 3))

# -------------------------------
# Save model & feature columns
# -------------------------------
joblib.dump(rf, "../models/rf_model.pkl")
print("✅ Model saved")

feature_columns = X.columns.tolist()
joblib.dump(feature_columns, "../models/feature_columns.pkl")
print("✅ Feature columns saved")

# Optional: show feature columns
print("Features used for training:")
print(feature_columns)

# Optional: show LabelEncoder classes
print("LabelEncoder classes (Crops):")
print(le.classes_)


✅ LabelEncoder saved
Random Forest MSE: 105.92
Random Forest R²: 0.977
✅ Model saved
✅ Feature columns saved
Features used for training:
['Month', 'Year', 'Rainfall', 'Crop', 'Season', 'Prev_Month_WPI', 'Temperature']
LabelEncoder classes (Crops):
['Bajra' 'Carrot' 'Corn' 'Cotton' 'Jowar' 'Masoor' 'Moong' 'Onion'
 'Orange' 'Rice' 'Soyabean' 'Sugarcane' 'Sunflower' 'SweetPotato' 'Tomato'
 'Urad' 'Wheat']


In [24]:
# Cell 3: Quick prediction with defaults

import pandas as pd
import joblib

# Load model and helpers
rf = joblib.load("../models/rf_model.pkl")
le = joblib.load("../models/label_encoder.pkl")
feature_columns = joblib.load("../models/feature_columns.pkl")

# Base prices for WPI → price/kg
base_prices = {
    "Bajra": 20, "Carrot": 25, "Corn": 18, "Cotton": 22, "Jowar": 21,
    "Masoor": 40, "Moong": 50, "Onion": 20, "Orange": 30, "Rice": 30,
    "Soyabean": 28, "Sugarcane": 10, "Sunflower": 45, "SweetPotato": 18,
    "Tomato": 25, "Urad": 48, "Wheat": 20
}
base_WPI = 100

# -------------------------------
# Default feature values
# -------------------------------
default_features = {
    "Rainfall": 50,
    "Season": 1,
    "Prev_Month_WPI": 100,
    "Temperature": 30
}

# -------------------------------
# Select crop (use index for simplicity)
# -------------------------------
print("Available crops:")
for i, c in enumerate(le.classes_):
    print(f"{i+1}. {c}")

crop_index = int(input("\nEnter crop number (1-17, default 17 for Wheat): ") or 17) - 1
crop_name = le.classes_[crop_index]

year_input = int(input("Enter Year (default 2026): ") or 2026)
month_input = int(input("Enter Month 1-12 (default 2): ") or 2)

# Prepare input with defaults
input_data = {
    "Month": month_input,
    "Year": year_input,
    "Rainfall": default_features["Rainfall"],
    "Crop": le.transform([crop_name])[0],
    "Season": default_features["Season"],
    "Prev_Month_WPI": default_features["Prev_Month_WPI"],
    "Temperature": default_features["Temperature"]
}

X_input = pd.DataFrame([input_data])
X_input = X_input[feature_columns]

# Predict WPI
predicted_wpi = rf.predict(X_input)[0]
print(f"\n✅ Predicted WPI for {crop_name}: {predicted_wpi:.2f}")

# Price per kg
if crop_name in base_prices:
    price_per_kg = (predicted_wpi / base_WPI) * base_prices[crop_name]
    print(f"Estimated price per kg: ₹{price_per_kg:.2f}")
else:
    print("❌ Crop not found in base_prices, cannot calculate price per kg")


Available crops:
1. Bajra
2. Carrot
3. Corn
4. Cotton
5. Jowar
6. Masoor
7. Moong
8. Onion
9. Orange
10. Rice
11. Soyabean
12. Sugarcane
13. Sunflower
14. SweetPotato
15. Tomato
16. Urad
17. Wheat

✅ Predicted WPI for Corn: 161.49
Estimated price per kg: ₹29.07
